# College Admission OpenEnv — TRL Training (Kaggle T4)

This notebook runs `train_trl_kaggle.py` end-to-end:
- Connects to the OpenEnv HTTP environment
- Trains a small model with TRL + QLoRA
- Compares trained vs random/untrained baselines
- Saves labeled plots and metrics
- Optional HF Hub / Space push


In [ ]:
!pip -q install "transformers>=4.46.0" "trl>=0.11.0" "peft>=0.13.0" \
                "accelerate>=0.34.0" "bitsandbytes>=0.43.0" "datasets>=2.20.0" \
                "huggingface_hub>=0.25.0" "pandas>=2.1.0" "matplotlib>=3.8.0" \
                "requests>=2.31.0"

In [ ]:
import os
import subprocess
import sys

# --- Required runtime config (local workspace, not Kaggle) ---
os.environ["COLLEGE_ENV_BASE_URL"] = "https://knight09-college-env.hf.space"
os.environ["BASE_MODEL_NAME"] = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
os.environ["OUTPUT_DIR"] = "./trl_runs/college_env_qwen25_15b_unsloth"

# --- W&B tracking (enabled) ---
os.environ["USE_WANDB"] = "1"
os.environ.setdefault("WANDB_PROJECT", "college-admission-openenv")
# os.environ["WANDB_API_KEY"] = "<your_wandb_api_key>"
# os.environ["WANDB_ENTITY"] = "<your_wandb_entity_optional>"

# --- Training/eval knobs ---
os.environ["TRAIN_EPISODES_PER_TEMPLATE"] = "110"
os.environ["MAX_TRAIN_STEPS"] = "600"
os.environ["TRAIN_BATCH_SIZE"] = "2"
os.environ["GRAD_ACCUM_STEPS"] = "8"
os.environ["EVAL_EPISODES_PER_TASK"] = "12"

# --- Optional HF Hub push ---
# os.environ["PUSH_TO_HUB"] = "1"
# os.environ["HF_TOKEN"] = "<your_hf_token>"
# os.environ["HUB_MODEL_REPO"] = "<username>/<model-repo-name>"

# --- Optional HF Space dashboard push ---
# os.environ["PUSH_TO_SPACE_DASHBOARD"] = "1"
# os.environ["HF_SPACE_REPO"] = "<username>/<space-repo-name>"

# UTF-8 on Windows avoids TRL template decoding issues.
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
cmd = [sys.executable, "-X", "utf8", "train_trl_kaggle.py"]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
TRAIN_OK = (result.returncode == 0)
print("TRAIN_OK:", TRAIN_OK)

: 

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

output_dir = Path(os.environ.get("OUTPUT_DIR", "./trl_runs/college_env_qwen25_05b"))
artifact_dir = output_dir / "artifacts"

summary_path = artifact_dir / "evaluation_summary.csv"
episodes_path = artifact_dir / "evaluation_episodes.csv"
print("Summary:", summary_path)
print("Episodes:", episodes_path)
if not summary_path.exists():
    print("\nNo artifacts found yet. Run the training cell above and ensure TRAIN_OK is True.")
else:
    print("\nEvaluation summary table:")
    display(pd.read_csv(summary_path).head(30))

for img_name in ["training_loss_curve.png", "episode_return_curve.png", "task_score_bar.png"]:
    img_path = artifact_dir / img_name
    print(f"\n{img_name} -> {img_path}")
    if img_path.exists():
        display(Image(filename=str(img_path)))


Summary: trl_runs\college_env_qwen25_05b\artifacts\evaluation_summary.csv
Episodes: trl_runs\college_env_qwen25_05b\artifacts\evaluation_episodes.csv

Evaluation summary table:
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\adity\miniconda3\Lib\site-packages\IPython\core\interactiveshell.py", line 3667, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\adity\AppData\Local\Temp\ipykernel_18056\2906639220.py", line 13, in <module>
    display(pd.read_csv(summary_path).head(30))
            ~~~~~~~~~~~^^^^^^^^^^^^^^
  File "c:\Users\adity\miniconda3\Lib\site-packages\pandas\io\parsers\readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "c:\Users\adity\miniconda3\Lib\site-packages\pandas\io\parsers\readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "c:\Users\adity\miniconda3\Lib\site-packages\pandas\io\parsers\readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "c:\Users\adity\miniconda3\Lib\site-pack